In [ ]:
# WESTERN BLOT AUSWERTUNG
# Relative Intensitäten und Signal-Rausch-Verhältnis (SNR)
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patches as mpatches
import pingouin as pg
import scikit_posthocs as sp_ph
import math
from matplotlib.ticker import AutoMinorLocator
from matplotlib.lines import Line2D


# ============================================================
# 1. DATEN EINLESEN
# ============================================================
# Jede CSV enthält Messungen einer Membran an einem bestimmten Tag.
# Namensschema: Datum + Membran-Nummer (z.B. "04.11 Membran 1")

# --- Messungen vom 04.11 (4 Membranen) ---
df04111 = pd.read_csv('04.11 Membran 1.csv', index_col=0)
df04111.columns = df04111.columns.str.strip()
df04112 = pd.read_csv('04.11 Membran 2.csv', index_col=0)
df04112.columns = df04112.columns.str.strip()
df04113 = pd.read_csv('04.11 Membran 3.csv', index_col=0)
df04113.columns = df04113.columns.str.strip()
df04114 = pd.read_csv('04.11 Membran 4.csv', index_col=0)
df04114.columns = df04114.columns.str.strip()

# --- Messungen vom 11.11 (4 Membranen) ---
df11111 = pd.read_csv('11.11 Membran 1.csv', index_col=0)
df11111.columns = df11111.columns.str.strip()
df11112 = pd.read_csv('11.11 Membran 2.csv', index_col=0)
df11112.columns = df11112.columns.str.strip()
df11113 = pd.read_csv('11.11 Membran 3.csv', index_col=0)
df11113.columns = df11113.columns.str.strip()
df11114 = pd.read_csv('11.11 Membran 4.csv', index_col=0)
df11114.columns = df11114.columns.str.strip()

# --- Messungen vom 15.10 und 09.12 (je eine Membran) ---
df15101 = pd.read_csv('15.10 Membran.csv', index_col=0)
df15101.columns = df15101.columns.str.strip()
df09121 = pd.read_csv('09.12 Membran.csv', index_col=0)
df09121.columns = df09121.columns.str.strip()


# StdDev CSV einmal einlesen
df_std = pd.read_csv('Results.csv', index_col=0)
df_std.columns = df_std.columns.str.strip()

# Jeder Membran den richtigen StdDev-Wert zuweisen
# Reihenfolge muss mit deiner CSV übereinstimmen!
df15101['StdDev'] = df_std.iloc[0]['StdDev']   # Zeile 1 → 15.10
df04111['StdDev'] = df_std.iloc[1]['StdDev']   # Zeile 2 → 04.11 M1
df04112['StdDev'] = df_std.iloc[2]['StdDev']   # Zeile 3 → 04.11 M2
df04113['StdDev'] = df_std.iloc[3]['StdDev']   # Zeile 4 → 04.11 M3
df04114['StdDev'] = df_std.iloc[4]['StdDev']   # Zeile 5 → 04.11 M4
df11111['StdDev'] = df_std.iloc[5]['StdDev']   # Zeile 6 → 11.11 M1
df11112['StdDev'] = df_std.iloc[6]['StdDev']   # Zeile 7 → 11.11 M2
df11113['StdDev'] = df_std.iloc[7]['StdDev']   # Zeile 8 → 11.11 M3
df11114['StdDev'] = df_std.iloc[8]['StdDev']   # Zeile 9 → 11.11 M4
df09121['StdDev'] = df_std.iloc[9]['StdDev']   # Zeile 10 → 09.12



# ============================================================
# 2. STRUKTUR DER DATEN
# ============================================================
# Jede CSV ist in Gruppen von 5 Zeilen aufgebaut (also jede 5 Zeile stellen eine Lane dar direkt nacheinenander):
#   Zeile 0: Lane       (Gesamtprotein der Spur)
#   Zeile 1: M1         (Marker 1 Signal)
#   Zeile 2: M23        (Marker 2/3 Signal)
#   Zeile 3: AQP4       (AQP4 Signal)
#   Zeile 4: Background (Hintergrundrauschen)
#
# Aus jeder Gruppe werden folgende Werte berechnet:
#   1) Hintergrundkorrektur: (Mean - Background) * Area
#   2) Relative Intensität:       Marker / Lane 

#Gruppengroeße, wie oben beschrieben:
group_size = 5


# ============================================================
# 3. BERECHNUNG RELATIVER INTENSITÄTEN (Marker / Lane)
# ============================================================
# Die hintergrundkorrigierten Signale werden auf die Lane
# normalisiert, um Unterschiede in der Beladung auszugleichen.
# Ergebnis: relativer Proteingehalt pro Spur

# Ergebnis-Dictionaries für jede Membran
results04111 = {'M1/Lane': [], 'M23/Lane': [], 'AQP4/Lane': []}
results04112 = {'M1/Lane': [], 'M23/Lane': [], 'AQP4/Lane': []}
results04113 = {'M1/Lane': [], 'M23/Lane': [], 'AQP4/Lane': []}
results04114 = {'M1/Lane': [], 'M23/Lane': [], 'AQP4/Lane': []}
results11111 = {'M1/Lane': [], 'M23/Lane': [], 'AQP4/Lane': []}
results11112 = {'M1/Lane': [], 'M23/Lane': [], 'AQP4/Lane': []}
results11113 = {'M1/Lane': [], 'M23/Lane': [], 'AQP4/Lane': []}
results11114 = {'M1/Lane': [], 'M23/Lane': [], 'AQP4/Lane': []}
results15101 = {'M1/Lane': [], 'M23/Lane': [], 'AQP4/Lane': []}
results09121 = {'M1/Lane': [], 'M23/Lane': [], 'AQP4/Lane': []}

# Anzahl der Replikate pro Membran (= Anzahl Zeilen / Gruppengröße, varriert zwischen 4 und 8)
n_groups04111 = len(df04111) // group_size
n_groups04112 = len(df04112) // group_size
n_groups04113 = len(df04113) // group_size
n_groups04114 = len(df04114) // group_size
n_groups11111 = len(df11111) // group_size
n_groups11112 = len(df11112) // group_size
n_groups11113 = len(df11113) // group_size
n_groups11114 = len(df11114) // group_size
n_groups15101 = len(df15101) // group_size
n_groups09121 = len(df09121) // group_size



# Erstellung einer Funktion, um die Berechnung für alle Membranen zu vereinfachen:
# Liest jede 5-Zeilen-Gruppe aus dem DataFrame,
# zieht den Hintergrund ab und normalisiert auf die Lane.
# Schreibt die Ergebnisse in das übergebene results-Dictionary.

def berechne_normalisierung(df, n_groups, results):
    for i in range(n_groups):
        block = df.iloc[i * group_size : (i + 1) * group_size]

        lane_area = block.iloc[0]['Area']
        lane_mean = block.iloc[0]['Mean']
        m1_area   = block.iloc[1]['Area']
        m1_mean   = block.iloc[1]['Mean']
        m23_area  = block.iloc[2]['Area']
        m23_mean  = block.iloc[2]['Mean']
        aqp4_area = block.iloc[3]['Area']
        aqp4_mean = block.iloc[3]['Mean']
        bg_mean   = block.iloc[4]['Mean']

        # Hintergrundkorrektur: (Messwert - Hintergrund) × Fläche
        lane_corr = (lane_mean - bg_mean) * lane_area
        m1_corr   = (m1_mean   - bg_mean) * m1_area
        m23_corr  = (m23_mean  - bg_mean) * m23_area
        aqp4_corr = (aqp4_mean - bg_mean) * aqp4_area

        # Normalisierung auf Lane (Beladungskontrolle)
        results['M1/Lane'].append(m1_corr   / lane_corr)
        results['M23/Lane'].append(m23_corr  / lane_corr)
        results['AQP4/Lane'].append(aqp4_corr / lane_corr)

# Einsetzen der Daten für die Normalisierung in die Funktion
# Normalisierung für alle Membranen berechnen:
berechne_normalisierung(df04111, n_groups04111, results04111)
berechne_normalisierung(df04112, n_groups04112, results04112)
berechne_normalisierung(df04113, n_groups04113, results04113)
berechne_normalisierung(df04114, n_groups04114, results04114)
berechne_normalisierung(df11111, n_groups11111, results11111)
berechne_normalisierung(df11112, n_groups11112, results11112)
berechne_normalisierung(df11113, n_groups11113, results11113)
berechne_normalisierung(df11114, n_groups11114, results11114)
berechne_normalisierung(df15101, n_groups15101, results15101)
berechne_normalisierung(df09121, n_groups09121, results09121)

# Ergebnisse ausgeben zur Kontrolle (ausgekommentiert um die Ausgabe nicht zu überladen):
# print("\nRelative Intensitäten (Marker / Lane):")
# for name, res in [('04111', results04111), ('04112', results04112),
#                  ('04113', results04113), ('04114', results04114),
#                  ('11111', results11111), ('11112', results11112),
#                  ('11113', results11113), ('11114', results11114),
#                  ('15101', results15101), ('09121', results09121)]:
#    print(f"\n  Membran {name}:")
#    for k, v in res.items():
#        print(f"    {k}: {[round(x, 4) for x in v]}")


# ============================================================
# 4. BOXPLOT ERSTELLEN
# ============================================================

# Um Wiederholungen zu vermeiden, wird das Layout einmalig
# als Funktion definiert und für alle 6 Boxplots genutzt.
# Funktion zum Erstellen von Boxplots mit einheitlichem Layout:
# Parameter:
#        data     : Daten mit berechneten Werten
#        labels   : x-Achsen-Beschriftung
#        ylabel   : Beschriftung der y-Achse
#        savename : Dateiname
def plot_boxplot(data, labels, ylabel, savename):

    plt.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': ['DejaVu Sans'],
        'font.size': 10,
        'axes.linewidth': 0.8,
        'axes.spines.top': False,
        'axes.spines.right': False,
        'xtick.direction': 'out',
        'ytick.direction': 'out',
        'xtick.major.size': 4,
        'ytick.major.size': 4,
        'xtick.major.width': 0.8,
        'ytick.major.width': 0.8,
        'pdf.fonttype': 42,
        'ps.fonttype': 42,
    })

    fig, ax = plt.subplots(figsize=(11, 6))

    # Boxplot zeichnen:
    bp = ax.boxplot(data,
                    tick_labels=labels,
                    patch_artist=True,
                    widths=0.45,
                    showmeans=False,
                    medianprops=dict(color='black', linewidth=1.5),
                    whiskerprops=dict(color='#333333', linewidth=0.8, linestyle='-'),
                    capprops=dict(color='#333333', linewidth=0.8),
                    boxprops=dict(linewidth=0.8),
                    flierprops=dict(marker='', markersize=0))

    # Boxen ohne Füllung:
    for patch in bp['boxes']:
        patch.set_facecolor('none')
        patch.set_edgecolor('#333333')
        patch.set_alpha(0.85)
        patch.set_linewidth(0.8)

    # Einzelne Datenpunkte als schwarze Punkte überlagern:
    for i, d in enumerate(data, start=1):
        ax.scatter([i] * len(d), d,
                   alpha=0.8, zorder=3,
                   color='black',
                   edgecolors='black',
                   s=5, linewidths=0.3)

    # Y-Achse: automatischer Bereich mit kleinem Puffer:
    all_points = [v for d in data for v in d]
    ymin = min(all_points)
    ymax = max(all_points)
    ax.set_ylim(ymin - ymax * 0.02, ymax * 1.004)

    # Gitternetz:
    ax.yaxis.set_minor_locator(AutoMinorLocator(2))
    ax.yaxis.grid(True, linestyle='--', linewidth=0.6, alpha=0.75, color='gray', which='major')
    ax.yaxis.grid(True, linestyle='--', linewidth=0.4, alpha=0.75, color='gray', which='minor')
    ax.xaxis.grid(False)
    ax.set_axisbelow(True)

    # Achsenbeschriftungen:
    ax.set_ylabel(ylabel, fontsize=10, labelpad=6)
    ax.set_xlabel('Membranen', fontsize=10, labelpad=10)
    ax.tick_params(axis='both', labelsize=9)
    ax.set_xlim(0.3, 10.7)

    # Hintergrund: 
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')

    plt.tight_layout()
    plt.savefig(f'{savename}.pdf', bbox_inches='tight')
    plt.savefig(f'{savename}.svg', bbox_inches='tight')
    plt.show()


# ============================================================
# 5. BOXPLOTABBILDUNGEN 1–3: RELATIVE INTENSITÄT (Marker / Lane)
# ============================================================
# Zeigt die hintergrundkorrigierten Signale auf
# Je ein Plot: M1, M23, AQP4 aller Membranen

labels = ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']

# Boxplot 1: M1 normalisiert auf Lane
plot_boxplot(
    data=[results15101['M1/Lane'], results04111['M1/Lane'], results04112['M1/Lane'],
          results04113['M1/Lane'], results04114['M1/Lane'], results11111['M1/Lane'],
          results11112['M1/Lane'], results11113['M1/Lane'], results11114['M1/Lane'],
          results09121['M1/Lane']],
    labels=labels,
    ylabel='Relative Intensität (a.u.)',
    savename='boxplot_publication1'
)

# Boxplot 2: M23 normalisiert auf Lane
plot_boxplot(
    data=[results15101['M23/Lane'], results04111['M23/Lane'], results04112['M23/Lane'],
          results04113['M23/Lane'], results04114['M23/Lane'], results11111['M23/Lane'],
          results11112['M23/Lane'], results11113['M23/Lane'], results11114['M23/Lane'],
          results09121['M23/Lane']],
    labels=labels,
    ylabel='Relative Intensität (a.u.)',
    savename='boxplot_publication2'
)

# Boxplot 3: AQP4 normalisiert auf Lane
plot_boxplot(
    data=[results15101['AQP4/Lane'], results04111['AQP4/Lane'], results04112['AQP4/Lane'],
          results04113['AQP4/Lane'], results04114['AQP4/Lane'], results11111['AQP4/Lane'],
          results11112['AQP4/Lane'], results11113['AQP4/Lane'], results11114['AQP4/Lane'],
          results09121['AQP4/Lane']],
    labels=labels,
    ylabel='Relative Intensität (a.u.)',
    savename='boxplot_publication3'
)


# ============================================================
# 6. BERECHNUNG DES SIGNAL-RAUSCH-VERHÄLTNISSES (SNR)
# ============================================================
# SNR = hintergrundkorrigiertes Signal / Hintergrundrauschen
# Gibt an, wie stark das Proteinsignal über dem Rauschen liegt.
# Höherer SNR = bessere Signalqualität

SNR15101 = {'M1/Lane': [], 'M23/Lane': [], 'AQP4/Lane': []}
SNR04111 = {'M1/Lane': [], 'M23/Lane': [], 'AQP4/Lane': []}
SNR04112 = {'M1/Lane': [], 'M23/Lane': [], 'AQP4/Lane': []}
SNR04113 = {'M1/Lane': [], 'M23/Lane': [], 'AQP4/Lane': []}
SNR04114 = {'M1/Lane': [], 'M23/Lane': [], 'AQP4/Lane': []}
SNR11111 = {'M1/Lane': [], 'M23/Lane': [], 'AQP4/Lane': []}
SNR11112 = {'M1/Lane': [], 'M23/Lane': [], 'AQP4/Lane': []}
SNR11113 = {'M1/Lane': [], 'M23/Lane': [], 'AQP4/Lane': []}
SNR11114 = {'M1/Lane': [], 'M23/Lane': [], 'AQP4/Lane': []}
SNR09121 = {'M1/Lane': [], 'M23/Lane': [], 'AQP4/Lane': []}

# Berechnet das Signal-Rausch-Verhältnis (SNR) für jedes Replikat.
# Funktion zur Berechnung des SNR für alle Membranen:

def berechne_snr(df, n_groups, snr_dict):
    
    for i in range(n_groups):
        block = df.iloc[i * group_size : (i + 1) * group_size]

        m1_area   = block.iloc[1]['Area']
        m1_mean   = block.iloc[1]['Mean']
        m23_area  = block.iloc[2]['Area']
        m23_mean  = block.iloc[2]['Mean']
        aqp4_area = block.iloc[3]['Area']
        aqp4_mean = block.iloc[3]['Mean']
        bg_mean   = block.iloc[4]['Mean']
        bg_std   = block.iloc[4]['StdDev']

        # Hintergrundkorrektur
        m1_corr   = (m1_mean   - bg_mean) * m1_area
        m23_corr  = (m23_mean  - bg_mean) * m23_area
        aqp4_corr = (aqp4_mean - bg_mean) * aqp4_area

        # SNR: korrigiertes Signal geteilt durch Hintergrundrauschen
        snr_dict['M1/Lane'].append(m1_corr   / bg_std)
        snr_dict['M23/Lane'].append(m23_corr  / bg_std)
        snr_dict['AQP4/Lane'].append(aqp4_corr / bg_std)


# SNR für alle Membranen berechnen durch Einsetzen der Daten in die Funktion:
berechne_snr(df15101, n_groups15101, SNR15101)
berechne_snr(df04111, n_groups04111, SNR04111)
berechne_snr(df04112, n_groups04112, SNR04112)
berechne_snr(df04113, n_groups04113, SNR04113)
berechne_snr(df04114, n_groups04114, SNR04114)
berechne_snr(df11111, n_groups11111, SNR11111)
berechne_snr(df11112, n_groups11112, SNR11112)
berechne_snr(df11113, n_groups11113, SNR11113)
berechne_snr(df11114, n_groups11114, SNR11114)
berechne_snr(df09121, n_groups09121, SNR09121)


# ============================================================
# 7. BOXPLOTABBILDUNGEN 4–6: SIGNAL-RAUSCH-VERHÄLTNIS (SNR)
# ============================================================
# Zeigt den SNR als Boxplot
# Je ein Plot: M1, M23, AQP4 für alle Membranen

# Boxplot 4: SNR für M1
plot_boxplot(
    data=[SNR15101['M1/Lane'], SNR04111['M1/Lane'], SNR04112['M1/Lane'],
          SNR04113['M1/Lane'], SNR04114['M1/Lane'], SNR11111['M1/Lane'],
          SNR11112['M1/Lane'], SNR11113['M1/Lane'], SNR11114['M1/Lane'],
          SNR09121['M1/Lane']],
    labels=labels,
    ylabel='Signal-Rausch-Verhältnis (a.u.)',
    savename='boxplot_publication4'
)

# Boxplot 5: SNR für M23
plot_boxplot(
    data=[SNR15101['M23/Lane'], SNR04111['M23/Lane'], SNR04112['M23/Lane'],
          SNR04113['M23/Lane'], SNR04114['M23/Lane'], SNR11111['M23/Lane'],
          SNR11112['M23/Lane'], SNR11113['M23/Lane'], SNR11114['M23/Lane'],
          SNR09121['M23/Lane']],
    labels=labels,
    ylabel='Signal-Rausch-Verhältnis (a.u.)',
    savename='boxplot_publication5'
)

# Boxplot 6: SNR für AQP4
plot_boxplot(
    data=[SNR15101['AQP4/Lane'], SNR04111['AQP4/Lane'], SNR04112['AQP4/Lane'],
          SNR04113['AQP4/Lane'], SNR04114['AQP4/Lane'], SNR11111['AQP4/Lane'],
          SNR11112['AQP4/Lane'], SNR11113['AQP4/Lane'], SNR11114['AQP4/Lane'],
          SNR09121['AQP4/Lane']],
    labels=labels,
    ylabel='Signal-Rausch-Verhältnis (a.u.)',
    savename='boxplot_publication6'
)
# ============================================================
# 8. DATEN IN LONG-FORMAT UMWANDELN (für statistische Tests)
# ============================================================
# pingouin erwartet einen DataFrame mit zwei Spalten:
#   'Wert'   -> der Messwert
#   'Gruppe' -> die Membran-Nummer (1–10)
# Umwandlungsfunktion:

def to_longformat(datenliste):
    rows = []
    for gruppe_idx, werte in enumerate(datenliste, start=1):
        for wert in werte:
            rows.append({'Gruppe': str(gruppe_idx), 'Wert': wert})
    return pd.DataFrame(rows)

# Relative Intensität (Marker / Lane)
dfresultsm1 = to_longformat([
    results15101['M1/Lane'], results04111['M1/Lane'], results04112['M1/Lane'],
    results04113['M1/Lane'], results04114['M1/Lane'], results11111['M1/Lane'],
    results11112['M1/Lane'], results11113['M1/Lane'], results11114['M1/Lane'],
    results09121['M1/Lane']])

dfresultsm23 = to_longformat([
    results15101['M23/Lane'], results04111['M23/Lane'], results04112['M23/Lane'],
    results04113['M23/Lane'], results04114['M23/Lane'], results11111['M23/Lane'],
    results11112['M23/Lane'], results11113['M23/Lane'], results11114['M23/Lane'],
    results09121['M23/Lane']])

dfresultsaqp4 = to_longformat([
    results15101['AQP4/Lane'], results04111['AQP4/Lane'], results04112['AQP4/Lane'],
    results04113['AQP4/Lane'], results04114['AQP4/Lane'], results11111['AQP4/Lane'],
    results11112['AQP4/Lane'], results11113['AQP4/Lane'], results11114['AQP4/Lane'],
    results09121['AQP4/Lane']])

# SNR
dfSNRm1 = to_longformat([
    SNR15101['M1/Lane'], SNR04111['M1/Lane'], SNR04112['M1/Lane'],
    SNR04113['M1/Lane'], SNR04114['M1/Lane'], SNR11111['M1/Lane'],
    SNR11112['M1/Lane'], SNR11113['M1/Lane'], SNR11114['M1/Lane'],
    SNR09121['M1/Lane']])

dfSNRm23 = to_longformat([
    SNR15101['M23/Lane'], SNR04111['M23/Lane'], SNR04112['M23/Lane'],
    SNR04113['M23/Lane'], SNR04114['M23/Lane'], SNR11111['M23/Lane'],
    SNR11112['M23/Lane'], SNR11113['M23/Lane'], SNR11114['M23/Lane'],
    SNR09121['M23/Lane']])

dfSNRaqp4 = to_longformat([
    SNR15101['AQP4/Lane'], SNR04111['AQP4/Lane'], SNR04112['AQP4/Lane'],
    SNR04113['AQP4/Lane'], SNR04114['AQP4/Lane'], SNR11111['AQP4/Lane'],
    SNR11112['AQP4/Lane'], SNR11113['AQP4/Lane'], SNR11114['AQP4/Lane'],
    SNR09121['AQP4/Lane']])

print("Daten AQP4 Lane Relative Intensität:",
      results09121['AQP4/Lane']
      )
print(
"Daten der AQP4 Lane SNR:",
        SNR09121['AQP4/Lane']
    )
# ============================================================
# 9. STATISTISCHE AUSWERTUNG
# ============================================================
# Kruskal-Wallis-Test: nicht-parametrischer Test auf Unterschiede zwischen den 10 Membranen bei SNR und der relativen Intensität.
# Bei signifikantem Ergebnis: Post-hoc Dunn's Test mit
# Bonferroni-Korrektur für paarweise Vergleiche.

for label, df in [('R_M1',    dfresultsm1),
                  ('R_M23',   dfresultsm23),
                  ('R_AQP4',  dfresultsaqp4),
                  ('SNR_M1',  dfSNRm1),
                  ('SNR_M23', dfSNRm23),
                  ('SNR_AQP4',dfSNRaqp4)]:
    print(f"\n{'─'*50}")
    print(f"Kruskal-Wallis-Test: {label}")
    print(pg.kruskal(data=df, dv='Wert', between='Gruppe'))

# Post-hoc Dunn's Test: paarweise Vergleiche mit Bonferroni-Korrektur
# Zeigt, welche Messgruppen sich signifikant voneinander unterscheiden
    print(f"Dunn's Test (Bonferroni): {label}")
    dunn = sp_ph.posthoc_dunn(df, val_col='Wert', group_col='Gruppe', p_adjust='bonferroni')
    print(dunn.round(4).to_string(float_format='{:.4f}'.format))
                                          







In [ ]:
# BRADFORD-ASSAY AUSWERTUNG
# Proteinkonzentrationsbestimmung via linearer Regression
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from matplotlib.ticker import AutoMinorLocator
from adjustText import adjust_text


# ============================================================
# 1. STANDARDKURVEN EINLESEN
# ============================================================

datacsv1 = pd.read_csv("PBS.csv", header=None,
                        names=['Absorption', 'Konzentration'], decimal=',')
datacsv2 = pd.read_csv("H2O.csv", header=None,
                        names=['Absorption', 'Konzentration'], decimal=',')

# print("Standardkurve PBS:\n", datacsv1)
# print("\nStandardkurve H2O:\n", datacsv2)


# ============================================================
# 2. LINEARE REGRESSION
# ============================================================

def lineare_regression(df, name):
    print(f"\n{'═'*60}")
    print(f"  Lineare Regression: {name}")
    print(f"{'═'*60}")

    y = df['Absorption']
    X = sm.add_constant(df['Konzentration'])
    model = sm.OLS(y, X).fit()
    print(model.summary())

    b0 = model.params['const']
    b1 = model.params['Konzentration']
    r2 = model.rsquared
    vorzeichen = '-' if b0 < 0 else '+'
    print(f"\n  Formel: y = {b1:.4f} · x {vorzeichen} {abs(b0):.4f}")
    print(f"  R²     = {r2:.4f}")
    return model

model_pbs = lineare_regression(datacsv1, 'PBS Standard')
model_h2o = lineare_regression(datacsv2, 'H2O Standard')


# ============================================================
# 3. ABSORPTIONSWERTE DER PROBEN DEFINIEREN
# ============================================================

proben_pbs = {
    '1072M PBS': {
        'included': [0.204533333333333, 0.142533333333333, 0.0840666666666667],
        'regionen': ['CTX', 'TH', 'HIP'],
        'excluded': []
    },
    '1073M PBS': {
        'included': [0.208766666666667, 0.101766666666667, 0.1466, 0.1187],
        'regionen': ['CTX', 'TH', 'HIP', 'CB'],
        'excluded': [0.00813333333333333],
        'regionen_excluded': ['OB']
    },
    '1074M PBS': {
        'included': [0.194, 0.196466666666667, 0.132066666666667, 0.104433333333333],
        'regionen': ['CTX', 'TH', 'HIP', 'CB'],
        'excluded': []
    },
    '1075M PBS': {
        'included': [0.261933333333333, 0.20920, 0.159866666666667,
                     0.0743666666666667, 0.0662333333333333],
        'regionen': ['CTX', 'TH', 'HIP', 'CB', 'OB'],
        'excluded': []
    },
    '1076M PBS': {
        'included': [0.152266666666667, 0.1463, 0.0878,
                     0.0845333333333334, 0.0592000000000001],
        'regionen': ['CTX', 'TH', 'HIP', 'CB', 'OB'],
        'excluded': []
    },
    '1077M PBS': {
        'included': [0.160166666666667, 0.116566666666667, 0.113266666666667, 0.0797],
        'regionen': ['CTX', 'TH', 'HIP', 'OB'],
        'excluded': [0.0244333333333333],
        'regionen_excluded': ['CB']
    },
    '1078M PBS': {
        'included': [0.223633333333333, 0.148166666666667,
                     0.129966666666667, 0.0909666666666667],
        'regionen': ['CTX', 'TH', 'HIP', 'CB'],
        'excluded': [-0.0302333333333334],
        'regionen_excluded': ['OB']
    },
    '1079M PBS': {
        'included': [0.241333333333333, 0.218066666666667, 0.1721, 0.153333333333333],
        'regionen': ['CTX', 'TH', 'HIP', 'CB'],
        'excluded': [0.0283333333333333],
        'regionen_excluded': ['OB']
    },
    '1080M PBS': {
        'included': [0.2628, 0.1888, 0.1165, 0.111966666666667],
        'regionen': ['CTX', 'TH', 'HIP', 'CB'],
        'excluded': [0.0435333333333333],
        'regionen_excluded': ['OB']
    },
    '1081M PBS': {
        'included': [0.183, 0.214133333333333, 0.127266666666667, 0.138, 0.0553],
        'regionen': ['CTX', 'TH', 'HIP', 'CB', 'OB'],
        'excluded': []
    },
    '1082M PBS': {
        'included': [0.2742, 0.307366666666667, 0.1302, 0.140433333333333, 0.0609666666666667],
        'regionen': ['CTX', 'TH', 'HIP', 'CB', 'OB'],
        'excluded': []
    },
}

proben_h2o = {
    '1072M H2O': {
        'included': [0.2757, 0.231533333333333, 0.0997],
        'regionen': ['CTX', 'TH', 'HIP'],
        'excluded': [0.0117],
        'regionen_excluded': ['CB']
    },
    '1073M H2O': {
        'included': [0.1947, 0.0532, 0.159633333333333, 0.111133333333333],
        'regionen': ['CTX', 'TH', 'HIP', 'CB'],
        'excluded': [0.0155],
        'regionen_excluded': ['OB']
    },
    '1074M H2O': {
        'included': [0.0896, 0.109766666666667, 0.0908, 0.0897666666666667],
        'regionen': ['CTX', 'TH', 'HIP', 'CB'],
        'excluded': []
    },
    '1075M H2O': {
        'included': [0.158266666666667, 0.0897333333333334,
                     0.0831666666666667, 0.0544, 0.0529666666666667],
        'regionen': ['CTX', 'TH', 'HIP', 'CB', 'OB'],
        'excluded': []
    },
    '1076M H2O': {
        'included': [0.1145, 0.1209, 0.0671666666666667],
        'regionen': ['CTX', 'TH', 'CB'],
        'excluded': [0.0406666666666667, 0.0307666666666667],
        'regionen_excluded': ['HIP', 'OB']
    },
    '1078M H2O': {
        'included': [0.129133333333333, 0.126033333333333, 0.109666666666667, 0.0922],
        'regionen': ['CTX', 'TH', 'HIP', 'CB'],
        'excluded': [0.0365333333333333],
        'regionen_excluded': ['OB']
    },
    '1079M H2O': {
        'included': [0.181166666666667, 0.152666666666667, 0.0877, 0.0717],
        'regionen': ['CTX', 'TH', 'HIP', 'CB'],
        'excluded': [0.0236333333333333],
        'regionen_excluded': ['OB']
    },
    '1080M H2O': {
        'included': [0.149333333333333, 0.1395, 0.0718333333333334, 0.0632666666666667],
        'regionen': ['CTX', 'TH', 'HIP', 'CB'],
        'excluded': [0.0267333333333334],
        'regionen_excluded': ['OB']
    },
    '1081M H2O': {
        'included': [0.175433333333333, 0.114466666666667, 0.0626333333333334, 0.0558],
        'regionen': ['CTX', 'TH', 'HIP', 'CB'],
        'excluded': [0.0374],
        'regionen_excluded': ['OB']
    },
    '1082M H2O': {
        'included': [0.167366666666667, 0.108966666666667,
                     0.0562666666666667, 0.0540333333333333],
        'regionen': ['CTX', 'TH', 'HIP', 'CB'],
        'excluded': [0.0234666666666666],
        'regionen_excluded': ['OB']
    },
}


# ============================================================
# 4. KONZENTRATIONEN BERECHNEN UND DATAFRAMES ERSTELLEN
# ============================================================

def berechne_konzentrationen(proben_dict, model, standard_name):
    b0 = model.params['const']
    b1 = model.params['Konzentration']

    print(f"\n{'═'*60}")
    print(f"  Konzentrationen [{standard_name}]")
    print(f"{'═'*60}")

    dataframes = {}
    for probenname, werte in proben_dict.items():
        print(f"\n  {probenname}:")
        rows = []
        for abs_wert in werte['included']:
            konz = (abs_wert - b0) / b1
            print(f"    {abs_wert:.4f} (included)  →  {konz:.4f} mg/ml")
            rows.append({'Absorption': abs_wert,
                         'Konzentration_mg_ml': konz,
                         'Status': 'included'})
        for abs_wert in werte['excluded']:
            konz = (abs_wert - b0) / b1
            print(f"    {abs_wert:.4f} (excluded)  →  {konz:.4f} mg/ml")
            rows.append({'Absorption': abs_wert,
                         'Konzentration_mg_ml': konz,
                         'Status': 'excluded'})
        dataframes[probenname] = pd.DataFrame(rows)
    return dataframes

dfs_pbs = berechne_konzentrationen(proben_pbs, model_pbs, 'PBS Standard')
dfs_h2o = berechne_konzentrationen(proben_h2o, model_h2o, 'H2O Standard')

# DataFrames PBS
df_1072M_PBS = dfs_pbs['1072M PBS']
df_1073M_PBS = dfs_pbs['1073M PBS']
df_1074M_PBS = dfs_pbs['1074M PBS']
df_1075M_PBS = dfs_pbs['1075M PBS']
df_1076M_PBS = dfs_pbs['1076M PBS']
df_1077M_PBS = dfs_pbs['1077M PBS']
df_1078M_PBS = dfs_pbs['1078M PBS']
df_1079M_PBS = dfs_pbs['1079M PBS']
df_1080M_PBS = dfs_pbs['1080M PBS']
df_1081M_PBS = dfs_pbs['1081M PBS']
df_1082M_PBS = dfs_pbs['1082M PBS']

# DataFrames H2O
df_1072M_H2O = dfs_h2o['1072M H2O']
df_1073M_H2O = dfs_h2o['1073M H2O']
df_1074M_H2O = dfs_h2o['1074M H2O']
df_1075M_H2O = dfs_h2o['1075M H2O']
df_1076M_H2O = dfs_h2o['1076M H2O']
df_1078M_H2O = dfs_h2o['1078M H2O']
df_1079M_H2O = dfs_h2o['1079M H2O']
df_1080M_H2O = dfs_h2o['1080M H2O']
df_1081M_H2O = dfs_h2o['1081M H2O']
df_1082M_H2O = dfs_h2o['1082M H2O']


# ============================================================
# 5. VERDÜNNUNGSBERECHNUNG FÜR H2O-PROBEN
# ============================================================
# Gemessene Konzentrationen stammen aus 1:10-Vorverdünnung.
# Schritt 1: Wahre Konzentration = gemessene Konzentration × 10
# Schritt 2: Verdünnung auf 1.2 mg/ml in 50 µl Endvolumen
#            nach C1 × V1 = C2 × V2:
#            V_Probe (µl) = (1.2 mg/ml × 50 µl) / wahre Konzentration
#            V_ddH2O (µl) = 50 - V_Probe
# Gilt für included UND excluded.
# Falls wahre Konzentration < 1.2 mg/ml → machbar = False, NaN

ZIEL_KONZ   = 1.2   # mg/ml
ENDVOLUMEN  = 50.0  # µl
VERDUENNUNG = 10    # Vorverdünnungsfaktor


def berechne_verduennung_h2o(proben_dict, model, standard_name):
    b0 = model.params['const']
    b1 = model.params['Konzentration']

    print(f"\n{'═'*60}")
    print(f"  Verdünnungsberechnung [{standard_name}]")
    print(f"  Vorverdünnung 1:{VERDUENNUNG}  |  "
          f"Ziel: {ZIEL_KONZ} mg/ml in {ENDVOLUMEN} µl")
    print(f"{'═'*60}")

    dataframes = {}

    for probenname, werte in proben_dict.items():
        rows = []
        print(f"\n  {probenname}:")
        print(f"  {'Region':<6} {'Status':<10} {'Abs':>8} "
              f"{'Konz_gem':>10} {'Konz_wahr':>11} "
              f"{'V_Probe':>9} {'V_ddH2O':>9}")
        print(f"  {'──────':<6} {'──────────':<10} {'────────':>8} "
              f"{'──────────':>10} {'───────────':>11} "
              f"{'─────────':>9} {'─────────':>9}")

        # Alle Werte verarbeiten: included und excluded
        alle_werte  = list(werte['included']) + list(werte.get('excluded', []))
        alle_reg    = (list(werte.get('regionen', [])) +
                       list(werte.get('regionen_excluded', [])))
        alle_status = (['included'] * len(werte['included']) +
                       ['excluded'] * len(werte.get('excluded', [])))

        for idx, (abs_wert, status) in enumerate(zip(alle_werte, alle_status)):
            region = alle_reg[idx] if idx < len(alle_reg) else f'R{idx+1}'

            # Gemessene Konzentration aus Regression
            konz_gem  = (abs_wert - b0) / b1

            # Wahre Konzentration: Vorverdünnung rückrechnen
            konz_wahr = konz_gem * VERDUENNUNG

            # Verdünnung auf Zielkonzentration
            if konz_wahr >= ZIEL_KONZ:
                v_probe = (ZIEL_KONZ * ENDVOLUMEN) / konz_wahr
                v_ddh2o = ENDVOLUMEN - v_probe
                machbar = True
            else:
                v_probe = np.nan
                v_ddh2o = np.nan
                machbar = False

            print(f"  {region:<6} {status:<10} {abs_wert:>8.4f} "
                  f"{konz_gem:>10.4f} {konz_wahr:>11.4f} "
                  f"{f'{v_probe:.2f}' if machbar else 'zu niedrig':>9} "
                  f"{f'{v_ddh2o:.2f}' if machbar else '—':>9}")

            rows.append({
                'Region':                  region,
                'Status':                  status,
                'Absorption':              round(abs_wert, 4),
                'Konz_gemessen_mg_ml':     round(konz_gem, 4),
                'Konz_wahr_mg_ml':         round(konz_wahr, 4),
                'V_Probe_ul':              round(v_probe, 2) if machbar else np.nan,
                'V_ddH2O_ul':              round(v_ddh2o, 2) if machbar else np.nan,
                'Endvolumen_ul':           ENDVOLUMEN,
                'Zielkonzentration_mg_ml': ZIEL_KONZ,
                'machbar':                 machbar,
            })

        dataframes[probenname] = pd.DataFrame(rows)

    return dataframes


# Verdünnungsberechnung für alle H2O-Proben
dfs_h2o_verduennung = berechne_verduennung_h2o(
    proben_h2o, model_h2o, 'H2O Standard')

# DataFrames als eigene Variablen
df_verduennung_1072M = dfs_h2o_verduennung['1072M H2O']
df_verduennung_1073M = dfs_h2o_verduennung['1073M H2O']
df_verduennung_1074M = dfs_h2o_verduennung['1074M H2O']
df_verduennung_1075M = dfs_h2o_verduennung['1075M H2O']
df_verduennung_1076M = dfs_h2o_verduennung['1076M H2O']
df_verduennung_1078M = dfs_h2o_verduennung['1078M H2O']
df_verduennung_1079M = dfs_h2o_verduennung['1079M H2O']
df_verduennung_1080M = dfs_h2o_verduennung['1080M H2O']
df_verduennung_1081M = dfs_h2o_verduennung['1081M H2O']
df_verduennung_1082M = dfs_h2o_verduennung['1082M H2O']


# ============================================================
# 6. REGRESSIONS-PLOT ERSTELLEN
# ============================================================

def plot_regression(df, model, name, eigene_absorption=None,
                    excluded_absorption=None, regionen=None,
                    regionen_excluded=None):

    plt.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': ['DejaVu Sans'],
        'font.size': 10,
        'axes.linewidth': 0.8,
        'axes.spines.top': False,
        'axes.spines.right': False,
        'xtick.direction': 'out',
        'ytick.direction': 'out',
        'xtick.major.size': 4,
        'ytick.major.size': 4,
        'pdf.fonttype': 42,
        'ps.fonttype': 42,
    })

    x  = df['Konzentration']
    y  = df['Absorption']
    b0 = model.params['const']
    b1 = model.params['Konzentration']
    r2 = model.rsquared

    inc_abs = list(eigene_absorption or [])
    exc_abs = list(excluded_absorption or [])
    inc_pos = [((a - b0) / b1, a) for a in inc_abs]
    exc_pos = [((a - b0) / b1, a) for a in exc_abs]
    alle_pos = inc_pos + exc_pos

    all_y_vals = [p[1] for p in alle_pos] + list(y)
    all_x_vals = [p[0] for p in alle_pos] + list(x)
    y_min  = min(all_y_vals)
    y_max  = max(all_y_vals)
    y_range = y_max - y_min if y_max != y_min else 0.1
    x_min  = min(all_x_vals)
    x_max  = max(all_x_vals)
    x_range = x_max - x_min if x_max != x_min else 1.0

    x_line = np.linspace(x.min(), x.max(), 100)
    y_line = b1 * x_line + b0

    fig, ax = plt.subplots(figsize=(7, 5.5))
    ax.set_ylim(y_min - y_range * 0.15, y_max + y_range * 0.40)
    ax.set_xlim(x_min - x_range * 0.08, x_max + x_range * 0.08)

    x_pred     = sm.add_constant(pd.Series(x_line, name='Konzentration'))
    prediction = model.get_prediction(x_pred)
    ci         = prediction.summary_frame(alpha=0.05)
    ax.fill_between(x_line, ci['mean_ci_lower'], ci['mean_ci_upper'],
                    alpha=0.15, color='#4363D8',
                    label='95% Konfidenzintervall', zorder=1)

    vorzeichen = '-' if b0 < 0 else '+'
    ax.plot(x_line, y_line, color='#4363D8', linewidth=1.2,
            label=f'y = {b1:.4f}·x {vorzeichen} {abs(b0):.4f}', zorder=2)
    ax.plot([], [], color='none', label=f'R² = {r2:.4f}')

    ax.scatter(x, y, color='#4363D8', edgecolors='#333333',
               s=40, zorder=3, linewidths=0.8, marker='s', label='Standards')

    texts    = []
    punkt_xs = []
    punkt_ys = []

    if inc_pos:
        for idx, (kx, ky) in enumerate(inc_pos):
            ax.scatter(kx, ky, color='#FFD700', edgecolors='#333333',
                       s=35, zorder=5, linewidths=0.8, marker='o')
            if regionen is not None and idx < len(regionen):
                t = ax.text(kx, ky, regionen[idx],
                            fontsize=7, color='#333333',
                            ha='center', va='bottom', zorder=10,
                            bbox=dict(boxstyle='round,pad=0.1',
                                      facecolor='white',
                                      edgecolor='none', alpha=0.7))
                texts.append(t)
                punkt_xs.append(kx)
                punkt_ys.append(ky)
        ax.scatter([], [], color='#FFD700', edgecolors='#333333',
                   s=35, linewidths=0.8, marker='o', label='Probe (inkludiert)')

    if exc_pos:
        for idx, (kx, ky) in enumerate(exc_pos):
            ax.scatter(kx, ky, color='#E6194B', edgecolors='#333333',
                       s=20, zorder=5, linewidths=0.5, marker='o')
            if regionen_excluded is not None and idx < len(regionen_excluded):
                t = ax.text(kx, ky, regionen_excluded[idx],
                            fontsize=7, color='#E6194B',
                            ha='center', va='bottom', zorder=10,
                            bbox=dict(boxstyle='round,pad=0.1',
                                      facecolor='white',
                                      edgecolor='none', alpha=0.7))
                texts.append(t)
                punkt_xs.append(kx)
                punkt_ys.append(ky)
        ax.scatter([], [], color='#E6194B', edgecolors='#333333',
                   s=20, linewidths=0.5, marker='o', label='Probe (exkludiert)')

    if texts:
        adjust_text(
            texts,
            x=punkt_xs,
            y=punkt_ys,
            ax=ax,
            expand=(2.5, 3.0),
            force_text=(1.5, 2.0),
            force_points=(1.0, 1.5),
            lim=500,
            arrowprops=dict(
                arrowstyle='-',
                color='#555555',
                lw=0.6,
                alpha=0.9,
                shrinkA=2,
                shrinkB=4,
            )
        )
        for t in texts:
            t.set_zorder(10)

    ax.set_xlabel('Konzentration (mg/ml)', fontsize=10, labelpad=5)
    ax.set_ylabel('Absorption [595 nm] (a.u.)', fontsize=10, labelpad=5)

    ax.xaxis.grid(True, linestyle='--', linewidth=0.6, alpha=0.75,
                  color='gray', which='major', zorder=0)
    ax.yaxis.set_minor_locator(AutoMinorLocator(2))
    ax.yaxis.grid(True, linestyle='--', linewidth=0.6, alpha=0.75,
                  color='gray', which='major', zorder=0)
    ax.yaxis.grid(True, linestyle='--', linewidth=0.4, alpha=0.75,
                  color='gray', which='minor', zorder=0)
    ax.set_axisbelow(True)

    ax.legend(fontsize=9, frameon=True, framealpha=0.9, edgecolor='#cccccc')
    fig.patch.set_facecolor('white')
    ax.set_facecolor('#f7f7f7')

    plt.tight_layout()
    plt.savefig(f'{name}_regression.svg', bbox_inches='tight')
    plt.savefig(f'{name}_regression.pdf', bbox_inches='tight')
    plt.show()
    plt.close()


# ============================================================
# 7. PLOTS FÜR ALLE PROBEN ERSTELLEN
# ============================================================

for probenname, werte in proben_pbs.items():
    dateiname = probenname.replace(' ', '_')
    plot_regression(datacsv1, model_pbs, dateiname,
                    eigene_absorption=werte['included'],
                    excluded_absorption=werte['excluded'],
                    regionen=werte.get('regionen'),
                    regionen_excluded=werte.get('regionen_excluded'))

for probenname, werte in proben_h2o.items():
    dateiname = probenname.replace(' ', '_')
    plot_regression(datacsv2, model_h2o, dateiname,
                    eigene_absorption=werte['included'],
                    excluded_absorption=werte['excluded'],
                    regionen=werte.get('regionen'),
                    regionen_excluded=werte.get('regionen_excluded'))

In [ ]:
# Dilution Calculator
# ====================
# Berechnet Sample- und Wasservolumen für eine Zielkonzentration.

# Formel:
#     C1 * V1 = C2 * V2
#     --> V1 (Sample) = (C2 * V_total) / C1
#     --> V2 (Water)  = V_total - V1

# Eingabe : CSV-Datei mit Spalten: Sample_ID, Region, Concentration_mg_mL
# Ausgabe : CSV-Datei + Konsolen-Ausgabe im gleichen Format wie LaTeX-Tabelle


import csv
import sys
import os

# ──────────────────────────────────────────────
# PARAMETER – hier anpassen
# ──────────────────────────────────────────────
TARGET_CONCENTRATION = 0.60   # mg/mL  → Zielkonzentration
TOTAL_VOLUME         = 50.00  # µL     → Gesamtvolumen pro Ansatz
INPUT_FILE           = "samples.csv"
OUTPUT_FILE          = "dilutions_output.csv"
# ──────────────────────────────────────────────


def calculate_dilution(concentration: float,
                       target: float = TARGET_CONCENTRATION,
                       total: float  = TOTAL_VOLUME) -> tuple[float, float]:
    """
    Gibt (sample_vol, water_vol) in µL zurück.
    Raises ValueError wenn Konzentration < Zielkonzentration.
    """
    if concentration < target:
        raise ValueError(
            f"Konzentration {concentration} mg/mL ist kleiner als "
            f"Zielkonzentration {target} mg/mL – Verdünnung nicht möglich."
        )
    sample_vol = round((target * total) / concentration, 2)
    water_vol  = round(total - sample_vol, 2)
    return sample_vol, water_vol


def process_csv(input_path: str, output_path: str) -> None:
    if not os.path.isfile(input_path):
        print(f"[FEHLER] Datei nicht gefunden: {input_path}")
        sys.exit(1)

    results = []
    warnings = []

    with open(input_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)

        # Spalten prüfen
        required = {"Sample_ID", "Region", "Concentration_mg_mL"}
        if not required.issubset(reader.fieldnames):
            missing = required - set(reader.fieldnames)
            print(f"[FEHLER] Fehlende Spalten in CSV: {missing}")
            sys.exit(1)

        for row in reader:
            sid    = row["Sample_ID"].strip()
            region = row["Region"].strip()
            conc   = row["Concentration_mg_mL"].strip().lower()

            if conc == "excluded" or conc == "":
                results.append({
                    "Sample_ID"          : sid,
                    "Region"             : region,
                    "Concentration_mg_mL": "excluded",
                    "Sample_uL"          : "---",
                    "Water_uL"           : "---",
                    "Status"             : "excluded"
                })
                continue

            try:
                conc_val = float(conc.replace(",", "."))
            except ValueError:
                warnings.append(f"  ! {sid} ({region}): Ungültiger Wert '{conc}' – übersprungen.")
                continue

            try:
                sample_vol, water_vol = calculate_dilution(conc_val)
                status = "ok"
            except ValueError as e:
                sample_vol, water_vol = None, None
                status = f"WARNUNG: {e}"
                warnings.append(f"  ! {sid} ({region}): {e}")

            results.append({
                "Sample_ID"          : sid,
                "Region"             : region,
                "Concentration_mg_mL": f"{conc_val:.2f}",
                "Sample_uL"          : f"{sample_vol:.2f}" if sample_vol is not None else "---",
                "Water_uL"           : f"{water_vol:.2f}"  if water_vol  is not None else "---",
                "Status"             : status
            })

    # ── CSV schreiben ──────────────────────────
    fieldnames = ["Sample_ID", "Region", "Concentration_mg_mL",
                  "Sample_uL", "Water_uL", "Status"]

    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)

    # ── Konsolen-Ausgabe ───────────────────────
    print("=" * 55)
    print(f"  Dilution Calculator")
    print(f"  Zielkonzentration : {TARGET_CONCENTRATION} mg/mL")
    print(f"  Gesamtvolumen     : {TOTAL_VOLUME} µL")
    print("=" * 55)

    current_id = None
    for r in results:
        if r["Sample_ID"] != current_id:
            current_id = r["Sample_ID"]
            print()

        if r["Status"] == "excluded":
            print(f"  {r['Sample_ID']} ({r['Region']}):")
            print(f"    --- excluded")
        elif r["Status"] == "ok":
            print(f"  {r['Sample_ID']} ({r['Region']}):")
            print(f"    Concentration [mg/mL] = {r['Concentration_mg_mL']}")
            print(f"    Sample        [µL]    = {r['Sample_uL']}")
            print(f"    Water         [µL]    = {r['Water_uL']}")
        else:
            print(f"  {r['Sample_ID']} ({r['Region']}): {r['Status']}")

    print()
    print("=" * 55)

    if warnings:
        print("\n[WARNUNGEN]")
        for w in warnings:
            print(w)

    print(f"\n[OK] Ergebnisse gespeichert: {output_path}")
    print(f"     {len([r for r in results if r['Status'] == 'ok'])} Proben berechnet, "
          f"{len([r for r in results if r['Status'] == 'excluded'])} excluded.")


if __name__ == "__main__":
    # Optionaler CLI-Aufruf: python dilution_calculator.py input.csv output.csv
    inp = sys.argv[1] if len(sys.argv) > 1 else INPUT_FILE
    out = sys.argv[2] if len(sys.argv) > 2 else OUTPUT_FILE
    process_csv(inp, out)


In [ ]:
# Erstellung der Abbildung für das Chemidoc Imaging System:

######################################################
## ChemiDoc_AF647_final.py                          ##
## ChemiDoc MP – Filterübersicht AF647              ##
######################################################

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

filters_name = [
    "460/60",   # Stain-Free Detektion
    "530/28", "590/110", "605/50", "647SP", "695/55", "730/30", "800/14"
]

filter_to_applications = {
    "460/60":  ["Stain-Free (UV-Emission)"],
    "530/28":  ["Alexa Fluor 488", "FITC", "Cy2", "SYBR Green"],
    "590/110": ["SYPRO Ruby", "GelRed", "EtBr", "Ponceau S"],
    "605/50":  ["Alexa Fluor 546", "Cy3", "DyLight 550", "Rhodamine"],
    "647SP":   ["Chemiluminescence (cutoff)"],
    "695/55":  ["Alexa Fluor 647", "Cy5", "DyLight 650"],
    "730/30":  ["Alexa Fluor 680", "IRDye 680RD", "DyLight 680"],
    "800/14":  ["IRDye 800CW", "Alexa Fluor 790", "DyLight 800"],
}

emission_nm    = 668   # AF647 Emission
excitation_nm  = 650   # AF647 Exzitation
uv_nm          = 302   # Stain-Free Anregung
sf_emission_nm = 450   # Stain-Free Emission

def filter_to_range(name):
    if "SP" in name:
        return 0, int(name.replace("SP", ""))
    c, w = map(int, name.split("/"))
    return c - w / 2, c + w / 2

ax_end = 860

fig, ax = plt.subplots(figsize=(15, 7.5))

for i, fname in enumerate(filters_name):
    w_min, w_max = filter_to_range(fname)
    highlight = fname == "695/55"
    color = "#e05c00" if highlight else "gray"
    lw    = 6         if highlight else 4

    ax.hlines(i, w_min, w_max, colors=color, linewidth=lw, zorder=3)

    # Filter-Name direkt neben der Linie, weißer Hintergrund im Vordergrund
    suffix = "  ★" if highlight else ""
    ax.text(w_max + 5, i, f"{fname}{suffix}",
            va='center', fontsize=10, clip_on=True, zorder=5,
            color="#e05c00" if highlight else "black",
            fontweight='bold' if highlight else 'normal',
            bbox=dict(facecolor='white', edgecolor='none', pad=1.5))

    apps = filter_to_applications.get(fname, [])
    if apps:
        in_range  = w_min <= emission_nm <= w_max
        sym       = '\u2713' if in_range else '\u2717'
        sym_color = 'green'  if in_range else 'red'
        ax.text(ax_end + 3, i, sym, va='center', fontsize=11,
                color=sym_color, clip_on=False)
        ax.text(ax_end + 18, i, ', '.join(apps), va='center', fontsize=10,
                color="#e05c00" if highlight else "black",
                fontweight='bold' if highlight else 'normal',
                clip_on=False)

# --- Vertikale Linien ---

# UV Stain-Free Anregung (lila)
ax.vlines(uv_nm, -0.5, len(filters_name),
          colors='purple', linewidth=1.5, linestyle='dashed', alpha=0.6)
ax.text(uv_nm + 3, len(filters_name) - 0.15,
        f"{uv_nm} nm\n(Stain-Free\nAnregung)", color='purple',
        fontsize=8, ha='left', va='top')

# Stain-Free Emission (türkis)
ax.vlines(sf_emission_nm, -0.5, len(filters_name),
          colors='#00897B', linewidth=1.5, linestyle='dashed', alpha=0.8)
ax.text(sf_emission_nm + 3, len(filters_name) - 0.15,
        f"{sf_emission_nm} nm\n(Stain-Free\nEmission)", color='#00897B',
        fontsize=8, ha='left', va='top')

# AF647 Exzitation (blau)
ax.vlines(excitation_nm, -0.5, len(filters_name),
          colors='#1a7abf', linewidth=1.5, linestyle='dashed', alpha=0.7)
ax.text(excitation_nm - 3, len(filters_name) - 0.15,
        f"{excitation_nm} nm\n(Exzitation)", color='#1a7abf',
        fontsize=8, ha='right', va='top')

# AF647 Emission (orange)
ax.vlines(emission_nm, -0.5, len(filters_name),
          colors='#e05c00', linewidth=1.5, linestyle='dashed', alpha=0.7)
ax.text(emission_nm + 3, len(filters_name) - 0.15,
        f"{emission_nm} nm\n(Emission)", color='#e05c00',
        fontsize=8, ha='left', va='top')

# --- Achse ---
ax.set_xlim(0, ax_end)
ax.set_xticks([0, 200, 400, 600, 800])
ax.set_xlabel(r"Wellenlänge [$nm$]", fontsize=12)
ax.set_ylabel("Filtertypen", fontsize=12)
ax.set_yticks([])

# --- Legende ---
handles = [
    plt.Line2D([0],[0], color='purple',  lw=1.5, linestyle='dashed', label='Stain-Free Anregung UV (302 nm)'),
    plt.Line2D([0],[0], color='#00897B', lw=1.5, linestyle='dashed', label='Stain-Free Emission (~450 nm) → 460/60'),
    plt.Line2D([0],[0], color='#1a7abf', lw=1.5, linestyle='dashed', label='AF647 Exzitation (650 nm)'),
    plt.Line2D([0],[0], color='#e05c00', lw=1.5, linestyle='dashed', label='AF647 Emission (668 nm)'),
    mpatches.Patch(color='#e05c00', label='Optimaler Filter: 695/55 ★'),
    plt.Line2D([0],[0], color='gray', linewidth=3, label='Weitere ChemiDoc-Filter'),
]
ax.legend(handles=handles, loc='upper left', fontsize=8.5, framealpha=0.95)

plt.tight_layout()
plt.savefig("ChemiDoc_AF647_final.png", dpi=300, bbox_inches='tight')
print("Gespeichert.")
